This notebook can be used to detect cars in images using YOLO. First we fine tune an Ultralytics YOLO model using the Stanford Cars dataset. Then we predict cars using the COCO validation dataset.

Mount disk access from Google Drive.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Download and format Standford Cars for fine tuning

Before running this notebook, Stanford Cars (https://www.kaggle.com/datasets/eduardo4jesus/stanford-cars-dataset) was downloaded and unzipped in Google Drive. The below code formats the raw archive so that it can be used to fine tune YOLO models from Ultralytics. We use a train/valid/test split of 80/10/10 with 6000 training and 600 validation/testing images.

In [ ]:
import os
import random
import shutil
from scipy.io import loadmat
import cv2

# Create Stanford Cars directory names
BASE_DIR = '/content/drive/MyDrive/stanfordcars/archive'
IMAGES_DIR = os.path.join(BASE_DIR, 'cars_train/cars_train')
ANNOT_DIR = os.path.join(BASE_DIR, 'car_devkit/devkit/cars_train_annos.mat')
OUTPUT_DIR = '/content/drive/MyDrive/stanfordcars_yolo'

# Ensure directories exist
os.makedirs(OUTPUT_DIR, exist_ok=True)
subdirs = ['train', 'val', 'test']
for s in subdirs:
    os.makedirs(f'{OUTPUT_DIR}/images/{s}', exist_ok=True)
    os.makedirs(f'{OUTPUT_DIR}/labels/{s}', exist_ok=True)

In [ ]:
# Read Stanford Cars annotations from disk
raw_annos = loadmat(ANNOT_DIR)['annotations'][0]
annos = []
for a in raw_annos:
    fname = a[5][0]
    src_img_path = os.path.join(IMAGES_DIR, fname)
    img = cv2.imread(src_img_path)
    annos.append((a, img.shape[:2]))

# Assert at least 7200 images loaded
if len(annos) < 7200:
    raise ValueError(f'Not enough images: {len(annos)}')

# Take random 7200 images
random.seed(42)
annos = random.sample(list(annos), 7200)
splits = {
    'train': annos[:6000],
    'val': annos[6000:6600],
    'test': annos[6600:]
}

In [ ]:
# Format one Stanford Cars annotation for Ultralytics
def convert_yolo_bbox(x1, y1, x2, y2, img_w, img_h):
    x_center = (x1 + x2) / 2.0 / img_w
    y_center = (y1 + y2) / 2.0 / img_h
    width = (x2 - x1) / img_w
    height = (y2 - y1) / img_h
    return f'0 {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}'

# Format all Stanford Cars annotations
for split_name, split_annos in splits.items():
    for a, (h, w) in split_annos:
        fname = a[5][0]
        x1, y1, x2, y2 = [int(a[i][0]) for i in range(0, 4)]
        src_img_path = os.path.join(IMAGES_DIR, fname)
        dst_img_path = os.path.join(OUTPUT_DIR, f'images/{split_name}/{fname}')
        label_path = os.path.join(OUTPUT_DIR, f'labels/{split_name}/{fname.replace(".jpg", ".txt")}')

        # Write label
        with open(label_path, 'w') as f:
            f.write(convert_yolo_bbox(x1, y1, x2, y2, w, h))

        # Write image
        shutil.copy(src_img_path, dst_img_path)

# Download and format COCO validation for testing

The below code downloads the COCO validation dataset. After fine tuning our YOLO model on Stanford Cars, we use make object detection predictions for cars. The COCO validation dataset is formated for use by Ultralytics.

In [2]:
!mkdir -p /content/coco2017
!wget -c http://images.cocodataset.org/zips/val2017.zip -O /content/coco2017/val2017.zip
!wget -c http://images.cocodataset.org/annotations/annotations_trainval2017.zip -O /content/coco2017/annotations_trainval2017.zip
!unzip -q /content/coco2017/val2017.zip -d /content/coco2017
!unzip -q /content/coco2017/annotations_trainval2017.zip -d /content/coco2017

--2025-06-06 00:16:31--  http://images.cocodataset.org/zips/val2017.zip
Resolving images.cocodataset.org (images.cocodataset.org)... 16.15.192.125, 54.231.197.201, 52.217.198.41, ...
Connecting to images.cocodataset.org (images.cocodataset.org)|16.15.192.125|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 815585330 (778M) [application/zip]
Saving to: ‘/content/coco2017/val2017.zip’

/content/coco2017/v 100%[===================>] 777.80M  17.4MB/s    in 49s     

2025-06-06 00:17:21 (15.7 MB/s) - ‘/content/coco2017/val2017.zip’ saved [815585330/815585330]

--2025-06-06 00:17:21--  http://images.cocodataset.org/annotations/annotations_trainval2017.zip
Resolving images.cocodataset.org (images.cocodataset.org)... 52.216.60.41, 52.217.195.105, 3.5.3.100, ...
Connecting to images.cocodataset.org (images.cocodataset.org)|52.216.60.41|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 252907541 (241M) [application/zip]
Saving to: ‘/content/coco201

In [6]:
from pycocotools.coco import COCO
import os
import shutil

# Create COCO validation directory names
coco_ann_file = '/content/coco2017/annotations/instances_val2017.json'
coco_images_dir = '/content/coco2017/val2017'
output_images_dir = '/content/drive/MyDrive/coco/images/val'
output_labels_dir = '/content/drive/MyDrive/coco/labels/val'

# Ensure directories exist
os.makedirs(output_images_dir, exist_ok=True)
os.makedirs(output_labels_dir, exist_ok=True)

In [7]:
coco = COCO(coco_ann_file)
car_cat_id = 3
all_img_ids = coco.getImgIds()

for img_id in all_img_ids:
    img_info = coco.loadImgs(img_id)[0]
    img_filename = img_info['file_name']
    img_width = img_info['width']
    img_height = img_info['height']

    # Copy car image to disk
    src_img_path = os.path.join(coco_images_dir, img_filename)
    dst_img_path = os.path.join(output_images_dir, img_filename)
    if not os.path.exists(dst_img_path):
        shutil.copy(src_img_path, dst_img_path)

    # Write formatted car annotations to disk
    ann_ids = coco.getAnnIds(imgIds=img_id, catIds=[car_cat_id], iscrowd=False)
    anns = coco.loadAnns(ann_ids)
    label_file = os.path.join(output_labels_dir, os.path.splitext(img_filename)[0] + '.txt')
    with open(label_file, 'w') as f:
        for ann in anns:
            bbox = ann['bbox']
            x, y, w, h = bbox
            x_center = (x + w / 2) / img_width
            y_center = (y + h / 2) / img_height
            w_norm = w / img_width
            h_norm = h / img_height
            f.write(f"0 {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}\n")

loading annotations into memory...
Done (t=1.01s)
creating index...
index created!


# Download Ultralytics

Ultralytics (https://docs.ultralytics.com/) maintains all versions of YOLO and we'll use it to build YOLO models.

In [2]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 64.1 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling 

# Fine tune our YOLO model

Write an Ultralytics configuration file that is used to fine tune a YOLO model using Stanford Cars. Our use case only wants to detect one class: car. After creating the configuration file, the below code writes it to disk in Google Drive.

In [22]:
%%writefile yolo.yaml
path: /content/drive/MyDrive/stanfordcars_yolo
train: images/train
val: images/val
nc: 1
names: ['car']

Writing yolo.yaml


In [ ]:
!mkdir -p /content/drive/MyDrive/stanfordcars_yolo/configs
!cp yolo.yaml /content/drive/MyDrive/stanfordcars_yolo/configs/

Fine tune a YOLO model using Stanford Cars!

In [26]:
from ultralytics import YOLO

model = YOLO('yolov8m.pt')
results = model.train(
    data='/content/drive/MyDrive/stanfordcars_yolo/configs/yolo.yaml',
    epochs=50,
    batch=16,
    optimizer='Adam',
    lr0=0.0003,
    lrf=0.01,
    weight_decay=0.00005,
    imgsz=640,
    name='yolov8m_cars',
    project='/content/drive/MyDrive/stanfordcars_yolo'
)

Ultralytics 8.3.151 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/stanfordcars_yolo/configs/yolo.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0003, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolov8m_cars, nbs=64, nms=False, opset=None, optimize=False, optimizer=Adam, overlap_mask=True, patience=100, perspective

train: Scanning /content/drive/MyDrive/stanfordcars_yolo/labels/train.cache... 6000 images, 0 backgrounds, 1 corrupt: 100%|██████████| 6000/6000 [00:00<?, ?it/s]

train: /content/drive/MyDrive/stanfordcars_yolo/images/train/07389.jpg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [     1.1476]
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


val: Fast image access ✅ (ping: 1.5±2.2 ms, read: 2.0±1.2 MB/s, size: 12.4 KB)


val: Scanning /content/drive/MyDrive/stanfordcars_yolo/labels/val.cache... 600 images, 0 backgrounds, 0 corrupt: 100%|██████████| 600/600 [00:00<?, ?it/s]


...

      50/50      7.03G     0.2452     0.1148     0.9056         15        640: 100%|██████████| 375/375 [03:23<00:00,  1.84it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 19/19 [00:08<00:00,  2.36it/s]

                   all        600        600          1          1      0.995      0.953



50 epochs completed in 2.996 hours.
Optimizer stripped from /content/drive/MyDrive/stanfordcars_yolo/yolov8m_cars/weights/last.pt, 52.0MB
Optimizer stripped from /content/drive/MyDrive/stanfordcars_yolo/yolov8m_cars/weights/best.pt, 52.0MB

Validating /content/drive/MyDrive/stanfordcars_yolo/yolov8m_cars/weights/best.pt...
Ultralytics 8.3.151 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 92 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 19/19 [00:11<00:00,  1.68it/s]


                   all        600        600          1          1      0.995      0.955
Speed: 0.2ms preprocess, 7.2ms inference, 0.0ms loss, 3.1ms postprocess per image
Results saved to /content/drive/MyDrive/stanfordcars_yolo/yolov8m_cars


Evaluate the fine tuned YOLO model. We reload the best weights from fine tuning from disk.

In [3]:
from ultralytics import YOLO

model = YOLO('/content/drive/MyDrive/stanfordcars_yolo/yolov8m_cars/weights/best.pt')
metrics = model.val()

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.3.151 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 92 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs


100%|██████████| 755k/755k [00:00<00:00, 24.3MB/s]


val: Fast image access ✅ (ping: 0.4±0.1 ms, read: 0.2±0.1 MB/s, size: 58.5 KB)


val: Scanning /content/drive/MyDrive/stanfordcars_yolo/labels/val.cache... 600 images, 0 backgrounds, 0 corrupt: 100%|██████████| 600/600 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 38/38 [00:12<00:00,  2.99it/s]


                   all        600        600          1          1      0.995      0.955
Speed: 0.4ms preprocess, 14.1ms inference, 0.0ms loss, 1.7ms postprocess per image
Results saved to runs/detect/val


# Test our YOLO model

Perform car detection using our fine tuned YOLO model. We reload the best weights from fine tuning from disk.

In [78]:
import os
import shutil
from ultralytics import YOLO

image_dir = '/content/drive/MyDrive/coco/images/val'
output_img_dir = '/content/drive/MyDrive/coco/car_only/images'

os.makedirs(output_img_dir, exist_ok=True)

In [80]:
model = YOLO('/content/drive/MyDrive/stanfordcars_yolo/yolov8m_cars/weights/best.pt')
conf = 0.9

for fname in os.listdir(image_dir):
    img_path = os.path.join(image_dir, fname)

    # Make predictions on image
    results = model.predict(source=img_path, conf=conf, verbose=False)
    boxes = results[0].boxes

    # Copy images with predicted cars
    if boxes is not None and len(boxes) > 0:
        class_ids = boxes.cls.cpu().numpy().astype(int)
        confs = boxes.conf.cpu().numpy()
        has_car = any(cls_id == 0 and conf >= conf for cls_id, conf in zip(class_ids, confs))
        if has_car:
            # Copy image
            shutil.copy(img_path, os.path.join(output_img_dir, fname))

Collect metrics on our YOLO model. Previously, we collected metrics to see how our model performed on Standford Cars, the data it was fine tuned on. Now, we collect metrics to see how it performs on the COCO validation dataset.

In [4]:
%%writefile metrics.yaml
path: /content/drive/MyDrive/coco
train: images/train
val: images/val
nc: 1
names: ['car']

Writing metrics.yaml


In [5]:
from ultralytics import YOLO

model = YOLO('/content/drive/MyDrive/stanfordcars_yolo/yolov8m_cars/weights/best.pt')
metrics = model.val(data='metrics.yaml')

Ultralytics 8.3.151 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 92 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.4±0.1 ms, read: 0.5±0.2 MB/s, size: 215.4 KB)


val: Scanning /content/drive/MyDrive/coco/labels/val.cache... 5000 images, 4465 backgrounds, 0 corrupt: 100%|██████████| 5000/5000 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 313/313 [03:52<00:00,  1.35it/s]


                   all       5000       1918     0.0715     0.0688     0.0283     0.0215
Speed: 0.2ms preprocess, 15.6ms inference, 0.0ms loss, 1.2ms postprocess per image
Results saved to runs/detect/val2


In [ ]:
# Copy prediction metrics to disk
!mkdir -p /content/drive/MyDrive/coco/runs
!cp -r /content/runs/ /content/drive/MyDrive/coco/runs